# XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
import joblib
import pickle
from sklearn.model_selection import train_test_split

In [3]:
df_load = pd.read_csv('data/drom_archive_cleaned_2018-2025full.csv')
df = df_load

```
Преобразование категориальных признаков
```

In [4]:
categorical_target = ['Город', 'Модель', 'Метка']      # Target Encoding
categorical_onehot = ['Тип двигателя', 'Коробка передач', 'Привод', 'Тип кузова']  # OneHot
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Возраст авто']

preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(), categorical_target),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_onehot),
        ('num', 'passthrough', numerical)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=2000, # количество деревьев
        n_jobs=-1, # задействует все ядра
        random_state=42,
        eval_metric='rmse' # метрика для обучения
    ))
])

```
Загрузка обучающей и тестовой выборок
```

In [5]:
y = df['Цена']
X = df.drop('Цена', axis=1)

In [6]:
df['price_stratum'] = pd.qcut(df['Цена'], q=10, labels=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['price_stratum'])

```
Обучение и сохранение модели
```

In [7]:
model.fit(X_train, y_train, regressor__verbose=True)
joblib.dump(model, "models/xgb_model.pkl")

['models/xgb_model.pkl']

```
Предсказание на тестовой выборке
```

In [8]:
y_pred = model.predict(X_test)

```
Оценка модели
```

In [9]:
xg_mse = mean_squared_error(y_test, y_pred)
xg_rmse = np.sqrt(xg_mse)
xg_mae = mean_absolute_error(y_test, y_pred)
xg_r2 = r2_score(y_test, y_pred)
xg_mape = mean_absolute_percentage_error(y_test, y_pred)

```
Вывод результатов
```

In [10]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)', 'Средняя абсолютная процентная ошибка (MAPE)'],
    'Результаты': [xg_mse, xg_rmse, xg_mae, xg_r2, xg_mape]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),24_883_006_484.92
1,Среднеквадратическая ошибка (RMSE),157_743.48
2,Средняя абсолютная ошибка (MAE),103_372.10
3,Коэффицент детерминации (R^2),0.92
4,Средняя абсолютная процентная ошибка (MAPE),0.10


```
Сохранение метрик в отдельный файл
```

In [11]:
metrics = {
    "model_name": "XGBoost",
    "MSE": xg_mse,
    "RMSE": xg_rmse,
    "MAE": xg_mae,
    "R2": xg_r2,
    "MAPE": xg_mape
}

In [12]:
with open("metrics/xgb_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)